## Overview

This notebook is intended to provide some snippets of code summarizing the distribution of DERs of various types across meters of various types, including the combinations in which DERs appear together behind meters.

## Setup

In [1]:
import getpass
import os
from dotenv import load_dotenv
import psycopg2
import pandas as pd
import numpy as np

**Connection**

Enter the EDM server address and the login credentials provided by Awesense. If you do not have the credentials, or have any trouble connecting, please contact api@awesense.com.
<span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [2]:
# Checks for Google Colab: If detected,
# it bypasses reading credentials from local files and securely draws from Colab Secrets.
try:
    from google.colab import userdata
    from google.colab.userdata import SecretNotFoundError
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from contextlib import suppress
    print('☁️ Running in Google Colab. Using Colab Secrets for SQLAPI connection')
    # Look for Colab Secrets, fall back to interactive prompts if missing
    with suppress(SecretNotFoundError): os.environ['EDM_HOST'] = userdata.get('EDM_HOST')
    with suppress(SecretNotFoundError): os.environ['EDM_USER'] = userdata.get('EDM_USER')
    with suppress(SecretNotFoundError): os.environ['EDM_PASSWORD'] = userdata.get('EDM_PASSWORD')
else:
    # If running locally, loads connection parameters from a `.env` file (in this directory or parent directories)
    load_dotenv()

# Prompt the user to manually enter any missing connection parameters not found in `.env` file / Google Colab Secrets
if 'EDM_HOST' not in os.environ: os.environ['EDM_HOST'] = input('EDM server address: ').strip()
if 'EDM_USER' not in os.environ: os.environ['EDM_USER'] = input('EDM username: ')

# psycopg2/libpq automatically read the PG* environment variables, so map the shared EDM_* names onto them.
# (The password is only mapped if provided via `.env`/Colab; otherwise libpq falls back to `~/.pgpass`.)
os.environ['PGHOST'] = os.environ['EDM_HOST']
os.environ['PGUSER'] = os.environ['EDM_USER']
os.environ.setdefault('PGDATABASE', 'edm')
if 'EDM_PASSWORD' in os.environ: os.environ['PGPASSWORD'] = os.environ['EDM_PASSWORD']

print('Verifying SQLAPI connection parameters/credentials with connection attempt')

try:
    # Test database connection
    conn = psycopg2.connect('')
    conn.close()
    print('✅ SQLAPI connection parameters/credentials verified')
except psycopg2.OperationalError:
    print('⚠️ SQLAPI credentials check failed (assume missing/incorrect password, but double-check `.env` / secrets!)')
    # Prompt user securely for password
    os.environ['PGPASSWORD'] = getpass.getpass('Enter EDM Password manually: ')

# Keeps `%sql` result tables rendering correctly on newer versions of prettytable
import prettytable
if 'DEFAULT' not in vars(prettytable): prettytable.DEFAULT = prettytable.TableStyle.DEFAULT

# Load the SQL extension and connect
get_ipython().run_line_magic('reload_ext', 'sql')
get_ipython().run_line_magic('sql', 'postgresql://')

Verifying SQLAPI connection parameters/credentials with connection attempt


✅ SQLAPI connection parameters/credentials verified


In [3]:
# PARAMETERS (Cell Tag: parameters)
# Do not split this cell structure. Papermill injects automated
# runtime overrides immediately below this block.
# If you don't want to be asked for input for these parameters, you can set them here.
grid_id = None  # Target grid identifier (e.g., 'GSO_4')

## DER combinations behind meters

**Input Parameters**

Enter the grid ID of interest.

The example below uses the GSO_4 grid.  

In [4]:
# Short-circuit check: If Papermill didn't inject values, prompt the user.
# To skip these prompts, set the parameter values directly in the parameters cell above (PAPERMILL PARAMETERS).
grid_id = grid_id or input("Enter grid ID: ").strip()  # GSO_4

**Retrieve all meter-DER pairs where the DER is downstream of (behind) the meter.**

In [5]:
%%sql meter_ders << SELECT ggs.grid_element_id as meter_id,
    ggs.meta ->> 'type_of_consumer' as meter_type,
    ge.type as der_type,
    ge.grid_element_id as der_id
FROM grid_element ge
JOIN grid_get_sources(:grid_id, ge.grid_element_id, True) ggs ON True
WHERE ge.grid_id = :grid_id AND ggs.type = 'Meter'
    AND (ge.type='EVCharger' OR ge.type='Photovoltaic' OR ge.type='Battery')
ORDER BY meter_type, meter_id;

 * postgresql://


34 rows affected.
Returning data to local variable meter_ders


In [6]:
# Convert the results to a data frame.
df_meter_ders = meter_ders.DataFrame()
df_meter_ders

,meter_id,meter_type,der_type,der_id
0,m_18332_21,residential,Photovoltaic,pv_18332_21
1,m_18561_21,residential,EVCharger,evch_18561_21
2,m_21139_21,residential,EVCharger,evch_21139_21
3,m_21275_21,residential,EVCharger,evch_21275_21
4,m_24302_21,residential,EVCharger,evch_24302_21
5,m_24473_21,residential,Photovoltaic,pv_24473_21
6,m_24831_21,residential,EVCharger,evch_24831_21
7,m_24958_21,residential,EVCharger,evch_24958_21
8,m_25158_21,residential,EVCharger,evch_25158_21
9,m_2542_21,residential,EVCharger,evch_2542_21


**Find the DER combinations**

In [7]:
# Pivotal the table by DER type
df_matrix = df_meter_ders.pivot(
    index=["meter_type", "meter_id"], columns="der_type", values="der_id"
)
df_matrix

der_type                    EVCharger Photovoltaic
meter_type  meter_id                              
residential m_18332_21            NaN  pv_18332_21
            m_18561_21  evch_18561_21          NaN
            m_21139_21  evch_21139_21          NaN
            m_21275_21  evch_21275_21          NaN
            m_24302_21  evch_24302_21          NaN
            m_24473_21            NaN  pv_24473_21
            m_24831_21  evch_24831_21          NaN
            m_24958_21  evch_24958_21          NaN
            m_25158_21  evch_25158_21          NaN
            m_2542_21    evch_2542_21          NaN
            m_27428_21            NaN  pv_27428_21
            m_28333_21  evch_28333_21          NaN
            m_28720_21  evch_28720_21          NaN
            m_29292_21  evch_29292_21          NaN
            m_29808_21  evch_29808_21          NaN
            m_30103_21            NaN  pv_30103_21
            m_32683_21  evch_32683_21          NaN
            m_34254_21            NaN  pv_34254_21
            m_34605_21  evch_34605_21          NaN
            m_36018_21  evch_36018_21          NaN
            m_36097_21  evch_36097_21          NaN
            m_37929_21            NaN  pv_37929_21
            m_39722_21            NaN  pv_39722_21
            m_40449_21  evch_40449_21          NaN
            m_40452_21            NaN  pv_40452_21
            m_43416_21  evch_43416_21          NaN
            m_43868_21  evch_43868_21          NaN
            m_46123_21  evch_46123_21          NaN
            m_46616_21            NaN  pv_46616_21
            m_49916_21  evch_49916_21          NaN
            m_50013_21  evch_50013_21          NaN
            m_7359_21    evch_7359_21          NaN
            m_7364_21    evch_7364_21          NaN
            m_7403_21    evch_7403_21          NaN

In [8]:
# Label the DER combinations
df_matrix["DERs Present"] = (
    (
        df_matrix["Battery"].isnull().map({True: "_", False: "B"})
        if np.any(df_matrix.columns == "Battery")
        else "_"
    )
    + (
        df_matrix["EVCharger"].isnull().map({True: "_", False: "E"})
        if np.any(df_matrix.columns == "EVCharger")
        else "_"
    )
    + (
        df_matrix["Photovoltaic"].isnull().map({True: "_", False: "P"})
        if np.any(df_matrix.columns == "Photovoltaic")
        else "_"
    )
)
df_matrix

der_type                    EVCharger Photovoltaic DERs Present
meter_type  meter_id                                           
residential m_18332_21            NaN  pv_18332_21          __P
            m_18561_21  evch_18561_21          NaN          _E_
            m_21139_21  evch_21139_21          NaN          _E_
            m_21275_21  evch_21275_21          NaN          _E_
            m_24302_21  evch_24302_21          NaN          _E_
            m_24473_21            NaN  pv_24473_21          __P
            m_24831_21  evch_24831_21          NaN          _E_
            m_24958_21  evch_24958_21          NaN          _E_
            m_25158_21  evch_25158_21          NaN          _E_
            m_2542_21    evch_2542_21          NaN          _E_
            m_27428_21            NaN  pv_27428_21          __P
            m_28333_21  evch_28333_21          NaN          _E_
            m_28720_21  evch_28720_21          NaN          _E_
            m_29292_21  evch_29292_21          NaN          _E_
            m_29808_21  evch_29808_21          NaN          _E_
            m_30103_21            NaN  pv_30103_21          __P
            m_32683_21  evch_32683_21          NaN          _E_
            m_34254_21            NaN  pv_34254_21          __P
            m_34605_21  evch_34605_21          NaN          _E_
            m_36018_21  evch_36018_21          NaN          _E_
            m_36097_21  evch_36097_21          NaN          _E_
            m_37929_21            NaN  pv_37929_21          __P
            m_39722_21            NaN  pv_39722_21          __P
            m_40449_21  evch_40449_21          NaN          _E_
            m_40452_21            NaN  pv_40452_21          __P
            m_43416_21  evch_43416_21          NaN          _E_
            m_43868_21  evch_43868_21          NaN          _E_
            m_46123_21  evch_46123_21          NaN          _E_
            m_46616_21            NaN  pv_46616_21          __P
            m_49916_21  evch_49916_21          NaN          _E_
            m_50013_21  evch_50013_21          NaN          _E_
            m_7359_21    evch_7359_21          NaN          _E_
            m_7364_21    evch_7364_21          NaN          _E_
            m_7403_21    evch_7403_21          NaN          _E_

In [9]:
# Pivot again to list and count the meters for each DER combination, grouped by meter type.
df_matrix.rename_axis(None, axis=1).reset_index().pivot_table(
    index=["meter_type"],
    columns="DERs Present",
    values="meter_id",
    aggfunc=lambda x: str(len(x)) + ": " + ", ".join(x.astype(str)),
)

DERs Present,_E_,__P
meter_type,,
residential,"25: m_18561_21, m_21139_21, m_21275_21, m_2430...","9: m_18332_21, m_24473_21, m_27428_21, m_30103..."
